# 🔌 Budujemy wlasny MCP Server z FastMCP

Wczoraj poznalismy MCP (Model Context Protocol) — standard laczenia modeli AI z narzedziami.
Dzis zbudujemy wlasny serwer MCP od zera!

Nasz serwer udostepni agentowi AI narzedzia IT helpdesk:
- 🔍 Wyszukiwanie w bazie wiedzy
- 📋 Sprawdzanie statusu zgloszen
- ➕ Tworzenie nowych zgloszen
- 🖥️ Sprawdzanie statusu systemow

```
Claude Desktop / Cursor / Agent
        │
        │ MCP Protocol
        ▼
┌─────────────────────────────┐
│  Nasz MCP Server (Python)  │
├─────────────────────────────┤
│  search_knowledge_base()   │
│  get_ticket()              │
│  create_ticket()           │
│  check_system_status()     │
└─────────────────────────────┘
```

**Czas:** ~30 minut | **Wymagania:** Python **3.10+** (Google Colab dziala od razu)

## 🛠️ 0. Instalacja FastMCP

### 📖 Wytlumaczenie:
**FastMCP** to biblioteka Pythona do tworzenia serwerow MCP.
Zamiast pisac boilerplate protokolu, piszesz zwykle funkcje Pythona z dekoratorami — FastMCP zajmie sie reszta.

Analogia: **FastMCP jest dla MCP tym, czym Flask jest dla HTTP.**

### 💡 Cwiczenie:
Sprawdz, jakie inne pakiety zainstalowal FastMCP: `!pip show fastmcp`

In [ ]:
import sys
print(f"Python: {sys.version}")
if sys.version_info < (3, 10):
    print("⚠️  FastMCP wymaga Python >= 3.10!")
    print("   W Google Colab dziala od razu (Python 3.10+).")
    print("   Lokalnie: uzyj nowszego Pythona, np.:")
    print("     python3.12 -m venv venv && source venv/bin/activate && pip install fastmcp")
    raise SystemExit("Wymagany Python >= 3.10")

!pip install -q fastmcp

from fastmcp import FastMCP
import json

print("✅ FastMCP zainstalowane!")

---
## 🔹 Krok 1: Pierwszy serwer MCP (~5 min)

### 📖 Wytlumaczenie:
Wczoraj widzielismy MCP od strony **konsumenta** — Claude Desktop uzywa narzedzi.
Dzis budujemy od strony **producenta** — TO TY definiujesz narzedzia.

Kazde narzedzie to zwykla funkcja Pythona z dekoratorem `@mcp.tool`.
FastMCP automatycznie:
- Generuje nazwe z nazwy funkcji
- Generuje opis z docstringa
- Generuje schemat parametrow z type hints

```python
@mcp.tool
def moje_narzedzie(query: str) -> str:
    """Ten docstring widzi model AI."""
    return wynik
```

### 💡 Cwiczenie:
Dodaj wlasne narzedzie `goodbye()` ktore zwraca pozegnanie.

In [ ]:
# === Tworzymy serwer MCP ===
mcp = FastMCP("IT Helpdesk")

# === Pierwsze narzedzie ===
@mcp.tool
def ping() -> str:
    """Sprawdz czy serwer MCP dziala."""
    return "🟢 Serwer MCP dziala!"

# Dekorator @mcp.tool rejestruje funkcje w serwerze,
# ale NIE zmienia jej dzialania — mozna ja wywolac normalnie:
print(ping())
print("\n✅ Serwer MCP 'IT Helpdesk' utworzony")

---
## 🔹 Krok 2: Narzedzia IT Helpdesk (~10 min)

### 📖 Wytlumaczenie:
Budujemy 4 narzedzia — te same, ktore mial nasz agent w Pythonie!
Ale teraz jako **MCP tools** — kazdy klient MCP (Claude Desktop, Cursor, dowolny agent) moze ich uzywac.

| # | Narzedzie | Co robi | Typ |
|---|-----------|---------|-----|
| 1 | `search_knowledge_base` | Szuka w runbookach IT | Read |
| 2 | `get_ticket` | Sprawdza status zgloszenia | Read |
| 3 | `create_ticket` | Tworzy nowe zgloszenie | Write |
| 4 | `check_system_status` | Sprawdza czy systemy dzialaja | Read |

### 💡 Cwiczenie:
Dodaj 5. narzedzie: `list_tickets()` ktore zwraca liste wszystkich zgloszen.

In [ ]:
# === Narzedzie 1: Baza wiedzy IT ===

KB_ARTICLES = {
    "drukarka": "RUNBOOK: Drukarki sieciowe — sprawdz ping do IP drukarki. Brak odpowiedzi = problem sieciowy. Odpowiada ale nie drukuje = sprzet.",
    "vpn": "RUNBOOK: VPN — sprawdz: 1) certyfikat wazny? 2) firewall? 3) split tunneling? Rozlaczanie co kilka minut = idle timeout.",
    "haslo": "RUNBOOK: Reset hasla — AD > ADUC > Reset Password. Synchronizacja z Azure AD do 30 min. Konto zablokowane po 5 probach = odblokuj w ADUC.",
    "outlook": "RUNBOOK: Outlook — 1) sprawdz profil (Control Panel > Mail), 2) wyczysc cache .ost, 3) sprawdz autodiscover.",
    "teams": "RUNBOOK: Teams — problemy z kamera: Settings > Privacy > Camera. Restart: zamknij w tray, wyczysc %appdata%/Microsoft/Teams.",
}

@mcp.tool
def search_knowledge_base(query: str) -> str:
    """Przeszukaj baze wiedzy IT (runbooki, procedury) po slowach kluczowych. Uzyj gdy uzytkownik pyta o rozwiazanie problemu technicznego."""
    query_lower = query.lower()
    results = [article for keyword, article in KB_ARTICLES.items() if keyword in query_lower]
    if results:
        return " | ".join(results)
    return f"Brak artykulow dla: '{query}'. Dostepne tematy: {', '.join(KB_ARTICLES.keys())}"

# Test
print("🔍 search_knowledge_base('problem z VPN'):")
print(search_knowledge_base("problem z VPN"))

In [ ]:
# === Narzedzie 2 i 3: Zgloszenia ===

TICKETS = {
    "T-001": {"text": "Laptop nie wlacza sie", "category": "Sprzet", "status": "Otwarty", "priority": "P1"},
    "T-002": {"text": "VPN rozlacza sie co 10 min", "category": "Siec", "status": "W trakcie", "priority": "P1"},
    "T-003": {"text": "Outlook nie synchronizuje", "category": "Oprogramowanie", "status": "Otwarty", "priority": "P2"},
    "T-004": {"text": "Konto zablokowane po 3 probach", "category": "Konto", "status": "Zamkniety", "priority": "P2"},
    "T-005": {"text": "Drukarka sieciowa nie odpowiada", "category": "Siec", "status": "Otwarty", "priority": "P2"},
}

_next_id = len(TICKETS) + 1

@mcp.tool
def get_ticket(ticket_id: str) -> str:
    """Sprawdz status zgloszenia IT po jego ID (np. T-001, T-002)."""
    ticket = TICKETS.get(ticket_id.upper())
    if not ticket:
        return f"Zgloszenie {ticket_id} nie znalezione. Dostepne: {', '.join(TICKETS.keys())}"
    return f"[{ticket_id}] {ticket['text']} | {ticket['category']} | {ticket['status']} | {ticket['priority']}"

@mcp.tool
def create_ticket(summary: str, category: str, priority: str) -> str:
    """Utworz nowe zgloszenie IT. Kategorie: Sprzet, Siec, Oprogramowanie, Konto. Priorytety: P1 (krytyczny) do P4 (niski)."""
    global _next_id
    ticket_id = f"T-{_next_id:03d}"
    TICKETS[ticket_id] = {"text": summary, "category": category, "status": "Otwarty", "priority": priority}
    _next_id += 1
    return f"✅ Utworzono {ticket_id}: {summary} [{category}, {priority}]"

# Test
print("📋 get_ticket('T-002'):")
print(get_ticket("T-002"))
print()
print("➕ create_ticket:")
print(create_ticket("Excel zawiesza sie przy duzych plikach", "Oprogramowanie", "P3"))

In [ ]:
# === Narzedzie 4: Status systemow ===

SYSTEMS = {
    "Active Directory": {"status": "🟢 Online", "uptime": "99.9%", "last_incident": "2026-01-15"},
    "Exchange Online": {"status": "🟢 Online", "uptime": "99.7%", "last_incident": "2026-02-01"},
    "VPN Gateway": {"status": "🟡 Degraded", "uptime": "98.2%", "last_incident": "2026-02-20"},
    "Jira": {"status": "🟢 Online", "uptime": "99.5%", "last_incident": "2026-01-28"},
    "Drukarki sieciowe": {"status": "🔴 Offline", "uptime": "95.1%", "last_incident": "2026-02-22"},
}

@mcp.tool
def check_system_status(system_name: str) -> str:
    """Sprawdz status systemu IT. Dostepne: Active Directory, Exchange Online, VPN Gateway, Jira, Drukarki sieciowe."""
    system = SYSTEMS.get(system_name)
    if not system:
        return f"System '{system_name}' nieznany. Dostepne: {', '.join(SYSTEMS.keys())}"
    return f"{system_name}: {system['status']} | Uptime: {system['uptime']} | Ostatni incydent: {system['last_incident']}"

# Test
print("🖥️ check_system_status('VPN Gateway'):")
print(check_system_status("VPN Gateway"))
print()
print("🖥️ check_system_status('Drukarki sieciowe'):")
print(check_system_status("Drukarki sieciowe"))

---
## 🔹 Krok 3: Testujemy przez protokol MCP (~10 min)

### 📖 Wytlumaczenie:
Do tej pory wywolywalismy narzedzia jako zwykle funkcje Pythona. To OK do debugowania,
ale **to nie jest MCP** — to po prostu Python.

Teraz uzyjesmy prawdziwego **klienta MCP**, ktory laczy sie z naszym serwerem
przez protokol MCP — dokladnie tak, jak robi to Claude Desktop.

FastMCP ma wbudowanego klienta `Client`, ktory moze polaczyc sie z serwerem **in-process**
(bez uruchamiania osobnego procesu). Zobaczysz:
- `list_tools()` — schematy narzedzi, ktore FastMCP wygenerował automatycznie
- `call_tool()` — wywolanie narzedzia przez protokol MCP

### 💡 Cwiczenie:
Wywolaj `call_tool` z wlasnym zapytaniem i sprawdz wynik.

In [ ]:
# === Prawdziwy test MCP — przez protokol, nie zwykle wywolanie funkcji ===

from fastmcp.client import Client

async def test_mcp_server():
    async with Client(mcp) as client:

        # 1. Jakie narzedzia udostepnia nasz serwer?
        tools = await client.list_tools()
        print("=" * 60)
        print(f"🔌 Serwer MCP: {len(tools)} narzedzi")
        print("=" * 60)
        for tool in tools:
            print(f"\n  📦 {tool.name}")
            print(f"     Opis: {tool.description}")
            print(f"     Parametry: {json.dumps(tool.inputSchema, ensure_ascii=False, indent=6)}")

        # 2. Wywolaj narzedzia przez MCP (tak jak robi Claude Desktop)
        print(f"\n{'=' * 60}")
        print("🧪 Test: wywolania przez protokol MCP")
        print("=" * 60)

        r1 = await client.call_tool("ping", {})
        print(f"\n1️⃣ ping → {r1.data}")

        r2 = await client.call_tool("search_knowledge_base", {"query": "vpn problemy"})
        print(f"\n2️⃣ search_knowledge_base('vpn problemy') → {r2.data[:100]}...")

        r3 = await client.call_tool("get_ticket", {"ticket_id": "T-001"})
        print(f"\n3️⃣ get_ticket('T-001') → {r3.data}")

        r4 = await client.call_tool("create_ticket", {
            "summary": "Serwer plikow niedostepny",
            "category": "Siec",
            "priority": "P1"
        })
        print(f"\n4️⃣ create_ticket → {r4.data}")

        r5 = await client.call_tool("check_system_status", {"system_name": "VPN Gateway"})
        print(f"\n5️⃣ check_system_status('VPN Gateway') → {r5.data}")

    print(f"\n{'=' * 60}")
    print("✅ Wszystkie narzedzia przetestowane przez protokol MCP!")
    print("   To dokladnie tak dziala, gdy Claude Desktop uzywa Twoich narzedzi.")

# W Jupyter/Colab mozna uzyc await bezposrednio:
await test_mcp_server()

### 🎯 Co wlasnie sie stalo?

Zauwaz roznice:
- **Wczesniej:** `search_knowledge_base("vpn")` — zwykle wywolanie funkcji Pythona
- **Teraz:** `client.call_tool("search_knowledge_base", {"query": "vpn"})` — wywolanie przez **protokol MCP**

To drugie to dokladnie to, co robi Claude Desktop, Cursor, lub dowolny inny klient MCP.
A `list_tools()` pokaze Ci **schematy JSON**, ktore FastMCP wygenerował automatycznie z Twoich type hints i docstringow.

**To jest sila MCP:** piszesz zwykle funkcje Pythona → FastMCP generuje schemat → dowolny klient AI moze ich uzywac.

---
## 🔹 Krok 4: Eksport jako serwer (~5 min)

### 📖 Wytlumaczenie:
Notebook to swietne miejsce do prototypowania. Ale zeby Claude Desktop (lub Cursor, lub inny klient MCP)
mogl uzywac naszych narzedzi, potrzebujemy samodzielnego pliku `.py`.

Ponizej generujemy:
1. **Plik serwera** `it_helpdesk_mcp.py` — gotowy do uruchomienia
2. **Konfiguracja Claude Desktop** — wklej do ustawien

### 💡 Cwiczenie:
Zabierz plik do domu i podlacz do Claude Desktop!
1. `pip install fastmcp`
2. Skopiuj `it_helpdesk_mcp.py` na swoj komputer
3. Dodaj config do Claude Desktop → restart → narzedzia dostepne!

In [ ]:
# === Konfiguracja Claude Desktop ===
# Plik serwera (it_helpdesk_mcp.py) jest dolaczony do materialow szkoleniowych.
# Ponizej konfiguracja do podlaczenia go do Claude Desktop.

config = {
    "mcpServers": {
        "it-helpdesk": {
            "command": "python",
            "args": ["PELNA_SCIEZKA/it_helpdesk_mcp.py"]
        }
    }
}

print("=" * 60)
print("📋 Konfiguracja Claude Desktop")
print("=" * 60)
print(json.dumps(config, indent=2))

print("\n📁 Plik konfiguracyjny:")
print("   Mac:     ~/Library/Application Support/Claude/claude_desktop_config.json")
print("   Windows: %APPDATA%/Claude/claude_desktop_config.json")

print("\n🚀 Kroki:")
print("   1. pip install fastmcp")
print("   2. Skopiuj it_helpdesk_mcp.py na swoj komputer")
print("   3. Zmien PELNA_SCIEZKA w ustawieniach na lokalizacje pliku")
print("   4. Restart Claude Desktop")
print("   5. Zapytaj Claude: Sprawdz status systemu VPN Gateway")

---
## 🎓 Podsumowanie

### Co zbudowalismy:
Kompletny **serwer MCP** z 5 narzedziami IT helpdesk:

| Narzedzie | Opis | Typ |
|-----------|------|-----|
| `ping` | Health check serwera | Read |
| `search_knowledge_base` | Wyszukiwanie w runbookach | Read |
| `get_ticket` | Status zgloszenia | Read |
| `create_ticket` | Tworzenie zgloszenia | Write |
| `check_system_status` | Status systemow IT | Read |

### Kluczowe lekcje:
1. **MCP tool = zwykla funkcja Pythona** z dekoratorem `@mcp.tool`
2. **FastMCP generuje schemat automatycznie** z type hints i docstrings
3. **Ten sam serwer dziala z kazdym klientem MCP** — Claude Desktop, Cursor, wlasny agent
4. **Wczoraj: konsument MCP. Dzis: producent MCP.** Pelne kolo.

### 💡 Cwiczenie koncowe:
Rozszerz serwer o narzedzie `escalate_ticket(ticket_id, reason)` ktore:
1. Zmienia priorytet zgloszenia na P1
2. Dodaje komentarz z powodem eskalacji
3. Zwraca potwierdzenie

> 🏆 *Wczoraj uzywaliscie narzedzi MCP. Dzis je budujecie. Jutro polaczysz je z systemami w swojej firmie.*